##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: JSON Mode Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/JSON_mode.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API can be used to generate a JSON output if you set the schema that you would like to use.

Two methods are available. You can either set the desired output in the prompt or supply a schema to the model separately.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/JSON_mode.ipynb).

### Install dependencies

In [ ]:
%pip install -U -q 'google-genai>=2.0.0'  # 2.0 for Interactions API

### Configure your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.

In [9]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

## Set your constrained output in the prompt

For this first example just describe the schema you want in the prompt itself, and ask the model to return JSON using `response_format`.

In [11]:
prompt = """
    List a few popular cookie recipes using this JSON schema:

    Recipe = {'recipe_name': str}
    Return: list[Recipe]
"""

Select the model you want to use in this guide:

In [13]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-preview", "gemini-3.1-flash-lite", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

raw_interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    response_format={"type": "text", "mime_type": "application/json"},
)

Parse the string to JSON:
> The response from `interactions.create` contains a list of `steps`. For text responses, access the output via `interaction.steps[-1].content[0].text`.


In [15]:
import json

response = json.loads(raw_interaction.steps[-1].content[0].text)
print(response)

[{'recipe_name': 'Chocolate Chip Cookies'}, {'recipe_name': 'Oatmeal Raisin Cookies'}, {'recipe_name': 'Peanut Butter Cookies'}, {'recipe_name': 'Sugar Cookies'}, {'recipe_name': 'Snickerdoodles'}]


For readability serialize and print it:

In [17]:
print(json.dumps(response, indent=4))

[
    {
        "recipe_name": "Chocolate Chip Cookies"
    },
    {
        "recipe_name": "Oatmeal Raisin Cookies"
    },
    {
        "recipe_name": "Peanut Butter Cookies"
    },
    {
        "recipe_name": "Sugar Cookies"
    },
    {
        "recipe_name": "Snickerdoodles"
    }
]


## Supply the schema to the model directly

The newest models allow you to pass a schema directly via the `response_format` parameter. The model output will then follow that schema.

In [19]:
import typing_extensions as typing

class Recipe(typing.TypedDict):
    recipe_name: str
    recipe_description: str
    recipe_ingredients: list[str]

For this example you want a list of `Recipe` objects, so pass the schema in the `response_format`.

In [21]:
result = client.interactions.create(
    model=MODEL_ID,
    input="List a few imaginative cookie recipes along with a one-sentence description as if you were a gourmet restaurant and their main ingredients",
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "recipe_name": {"type": "string"},
                    "recipe_description": {"type": "string"},
                    "recipe_ingredients": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                },
                "required": ["recipe_name", "recipe_description", "recipe_ingredients"],
            },
        },
    },
)

In [22]:
print(json.dumps(json.loads(result.steps[-1].content[0].text), indent=4))

[
    {
        "recipe_name": "Lavender & Wildflower Honey Shortbread",
        "recipe_description": "A delicate, buttery shortbread infused with hand-harvested culinary lavender and finished with a glaze of artisanal wildflower honey.",
        "recipe_ingredients": [
            "European-style butter",
            "all-purpose flour",
            "granulated sugar",
            "dried lavender buds",
            "wildflower honey",
            "Maldon sea salt"
        ]
    },
    {
        "recipe_name": "Umami Miso & White Chocolate Chunk",
        "recipe_description": "A nuanced exploration of sweet and savory notes blending fermented white miso with velvety white chocolate and toasted macadamia nuts.",
        "recipe_ingredients": [
            "White miso paste",
            "white chocolate chunks",
            "toasted macadamia nuts",
            "brown butter",
            "flour",
            "cane sugar"
        ]
    },
    {
        "recipe_name": "Smoked Sea Salt 

It is the recommended method if you're using a compatible model.

## Next Steps
### Useful API references:

Check the [structured output](https://ai.google.dev/gemini-api/docs/structured-output) documentation or the [`GenerationConfig`](https://ai.google.dev/api/generate-content#generationconfig) API reference for more details

### Related examples

* The constrained output is used in the [Text summarization](../examples/json_capabilities/Text_Summarization.ipynb) example to provide the model a format to summarize a story (genre, characters, etc...)
* The [Object detection](../examples/Object_detection.ipynb) examples are using the JSON constrained output to uniformize the output of the detection.

### Continue your discovery of the Gemini API

JSON is not the only way to constrain the output of the model, you can also use an [Enum](../quickstarts/Enum.ipynb). [Function calling](../quickstarts/Function_calling.ipynb) and [Code execution](../quickstarts/Code_Execution.ipynb) are other ways to enhance your model by either using your own functions or by letting the model write and run them.